# Claims Anomaly Detection — Data Exploration & Cleaning

**Track 3: Claims Management & Fraud Detection** — openIMIS Hackathon

This notebook covers Phases 1 and 2 of the technical plan, run incrementally so you can validate each transformation as you go.

**Order of work:**
1. Load and inspect the raw CSV
2. Anonymise (drop direct identifiers, hash member numbers)
3. Clean dates and numeric columns
4. Explore distributions and key statistics
5. Engineer the 7 model features
6. Compute the proxy fraud label
7. Save outputs for the next phase (model training)

Once everything works here, the logic moves into clean `.py` scripts (`anonymise.py`, `explore.py`, `features.py`) for the actual hackathon submission.

**Privacy:** the raw CSV contains real patient names. Do not commit it. The `.gitignore` should already exclude `*RAW*.csv` and `OVERALL_CLAIMS_PAID_ANALYSIS*.csv`.

## 0. Setup

Imports and config. Run this once at the start of every session.

In [ ]:
import hashlib
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")

# Paths — adjust if your file is elsewhere
RAW_PATH         = "OVERALL_CLAIMS_PAID_ANALYSIS_-_AUDIT_DATE_-_202507-11.csv"
ANONYMISED_PATH  = "data/claims_anonymised.csv"
FEATURES_PATH    = "data/claims_features.csv"
FULL_OUTPUT_PATH = "data/claims_full.csv"
PLOTS_DIR        = "plots"

os.makedirs("data", exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Load the raw CSV

First look — shape, columns, dtypes, a few sample rows. Don't transform anything yet.

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# Column names and dtypes
print("COLUMNS:")
for col in df.columns:
    print(f"  {col:25s}  {df[col].dtype}")

In [ ]:
# Quick null check — which columns have missing data?
nulls = df.isnull().sum()
nulls[nulls > 0].sort_values(ascending=False)

**Observation to confirm before continuing:**
- `DOCTOR` is 100% null — drop or ignore in features.
- `CLAIM AUDIT DATE` is partly null (~38%) — don't rely on it for date logic.
- `DIAGNOSIS CODE` and `ICD CODE` have ~9K and ~5K nulls respectively.

## 2. Anonymise the dataset

Drop direct identifiers, hash the membership number so per-member patterns stay trackable. This is the **only** version that can ever be committed to the repository.

In [ ]:
# Before: peek at what we're removing (to verify these columns really do contain PII)
pii_cols = ["PATIENT NAME", "PRINCIPAL MEMBER", "MEMBERSHIP NO"]
df[pii_cols].head(5)

In [ ]:
# Drop direct identifiers
df = df.drop(columns=["PATIENT NAME", "PRINCIPAL MEMBER"])

# Hash MEMBERSHIP NO — keeps per-member grouping possible without exposing the real ID
df["MEMBER_HASH"] = df["MEMBERSHIP NO"].apply(
    lambda x: hashlib.sha256(str(x).encode()).hexdigest()[:12]
    if pd.notna(x) else "UNKNOWN"
)
df = df.drop(columns=["MEMBERSHIP NO"])

print(f"Anonymised shape: {df.shape}")
print("Sample MEMBER_HASH values:")
df["MEMBER_HASH"].head(3).to_list()

In [ ]:
# Save the anonymised version — this is the version that goes into the repo
df.to_csv(ANONYMISED_PATH, index=False)
print(f"Saved → {ANONYMISED_PATH}")
print(f"Size : {os.path.getsize(ANONYMISED_PATH)/1e6:.1f} MB")

## 3. Clean dates

All three date columns arrive as strings. Convert them to proper datetime objects and check how many failed to parse.

In [ ]:
# Inspect raw date format first
df[["CLAIM DATE", "SERVICE DATE", "CLAIM AUDIT DATE"]].head(5)

In [ ]:
for col in ["CLAIM DATE", "SERVICE DATE", "CLAIM AUDIT DATE"]:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
    failed = df[col].isna().sum()
    print(f"  {col:20s}: {failed:>7,} failed to parse (now NaT)")

print()
print(f"Date range (CLAIM DATE)  : {df['CLAIM DATE'].min().date()} → {df['CLAIM DATE'].max().date()}")
print(f"Date range (SERVICE DATE): {df['SERVICE DATE'].min().date()} → {df['SERVICE DATE'].max().date()}")

## 4. Clean numeric columns

Amounts may have commas, KES prefixes, or whitespace. Strip them and convert to numeric. Anything that still fails to parse becomes NaN.

In [ ]:
# Peek at the raw amounts to see what cleaning is needed
df[["INVOICE AMOUNT", "SETTLED AMOUNT", "INV AMOUNT", "PAYABLE AMOUNT"]].head(5)

In [ ]:
for col in ["INVOICE AMOUNT", "SETTLED AMOUNT", "INV AMOUNT", "PAYABLE AMOUNT"]:
    df[col] = (
        df[col].astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("KES", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["INVOICE AMOUNT", "SETTLED AMOUNT", "INV AMOUNT", "PAYABLE AMOUNT"]].describe()

## 5. Key statistics

Now that the data is clean, compute the numbers needed for the README, the demo pitch, and the model parameters.

In [ ]:
# Financial totals
total_invoice  = df["INVOICE AMOUNT"].sum()
total_settled  = df["SETTLED AMOUNT"].sum()
total_variance = total_invoice - total_settled

print("FINANCIAL TOTALS")
print(f"  Total invoiced  : KES {total_invoice:>15,.2f}")
print(f"  Total settled   : KES {total_settled:>15,.2f}")
print(f"  Total variance  : KES {total_variance:>15,.2f} ({total_variance/total_invoice:.1%})")

In [ ]:
# Claim lag
df["claim_lag_days"] = (df["CLAIM DATE"] - df["SERVICE DATE"]).dt.days
df["claim_lag_days"] = df["claim_lag_days"].clip(lower=0)

print("CLAIM LAG (days between service and submission)")
print(df["claim_lag_days"].describe())
print()
print(f"  Claims with lag > 90 days  : {(df['claim_lag_days'] > 90).sum():>7,}")
print(f"  Claims with lag > 180 days : {(df['claim_lag_days'] > 180).sum():>7,}")
print(f"  Max lag                    : {df['claim_lag_days'].max():>7,.0f} days")

In [ ]:
# Invoice inflation
df["inflation_ratio_raw"] = df["INVOICE AMOUNT"] / df["SETTLED AMOUNT"].replace(0, pd.NA)

print("INVOICE INFLATION RATIO (invoice ÷ settled)")
print(df["inflation_ratio_raw"].describe())
print()
print(f"  Claims with inflation > 3x : {(df['inflation_ratio_raw'] > 3).sum():,}")

In [ ]:
# Proxy fraud label — used later for evaluation
suspicious_mask = df["SETTLED AMOUNT"] < df["INVOICE AMOUNT"] * 0.80

print("PROXY FRAUD LABEL (settled < 80% of invoice)")
print(f"  Suspicious : {suspicious_mask.sum():>7,} ({suspicious_mask.mean():.1%})")
print(f"  Clean      : {(~suspicious_mask).sum():>7,}")
print()
print(f"  → Use contamination={suspicious_mask.mean():.2f} in Isolation Forest")

In [ ]:
# Duplicates — strongest deterministic anomaly
dupes = df[df.duplicated("CLAIM NO", keep=False)]
varying_amounts = dupes.groupby("CLAIM NO")["INVOICE AMOUNT"].nunique()
real_dupes = (varying_amounts > 1).sum()

print("DUPLICATE CLAIM NOs")
print(f"  Total rows sharing a CLAIM NO       : {len(dupes):>7,}")
print(f"  CLAIM NOs with differing amounts    : {real_dupes:>7,}")

In [ ]:
# Status breakdown
df["STATUS"].value_counts()

In [ ]:
# Benefit code breakdown
df["BENEFIT CODE"].value_counts().head(10)

In [ ]:
# Unique counts
print(f"Providers : {df['PROVIDER NAME'].nunique():>7,}")
print(f"Members   : {df['MEMBER_HASH'].nunique():>7,}")
print(f"ICD codes : {df['ICD CODE'].nunique():>7,}")
print(f"Claim types: {df['CLAIM TYPE'].unique().tolist()}")

## 6. Exploratory plots

Visual confirmation of what the numbers above said. Saved to `plots/` so they can be reused in the README.

In [ ]:
# Plot 1 — invoice amount distribution (log scale, since data is heavily right-skewed)
valid = df["INVOICE AMOUNT"].dropna()
valid = valid[valid > 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log10(valid), bins=60, color="#1976D2", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Invoice Amount (log₁₀ KES)", fontsize=12)
ax.set_ylabel("Number of Claims", fontsize=12)
ax.set_title("Distribution of Invoice Amounts (log scale)", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"KES {10**x:,.0f}"))
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/invoice_distribution.png", dpi=150)
plt.show()

In [ ]:
# Plot 2 — claim lag distribution with rule thresholds marked
lag_data = df["claim_lag_days"].dropna()
lag_data = lag_data[lag_data <= 400]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(lag_data, bins=80, color="#388E3C", edgecolor="white", linewidth=0.4)
ax.axvline(90,  color="#D32F2F", linestyle="--", linewidth=1.5, label="90-day rule")
ax.axvline(180, color="#F57C00", linestyle="--", linewidth=1.5, label="180-day extreme")
ax.set_xlabel("Days between service and claim submission", fontsize=12)
ax.set_ylabel("Number of Claims", fontsize=12)
ax.set_title("Claim Lag Distribution (capped at 400 days)", fontsize=14, fontweight="bold")
ax.legend()
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/claim_lag_distribution.png", dpi=150)
plt.show()

In [ ]:
# Plot 3 — invoice vs settled scatter, highlighting suspicious claims
sample = df[["INVOICE AMOUNT", "SETTLED AMOUNT"]].dropna().sample(
    n=min(15000, len(df)), random_state=42
)
mask = sample["SETTLED AMOUNT"] < sample["INVOICE AMOUNT"] * 0.80

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sample.loc[~mask, "INVOICE AMOUNT"], sample.loc[~mask, "SETTLED AMOUNT"],
           alpha=0.15, s=5, color="#1976D2", label="Normal")
ax.scatter(sample.loc[mask, "INVOICE AMOUNT"], sample.loc[mask, "SETTLED AMOUNT"],
           alpha=0.4, s=8, color="#D32F2F", label="Suspicious (settled < 80%)")
max_val = min(sample["INVOICE AMOUNT"].quantile(0.99), 500000)
ax.plot([0, max_val], [0, max_val], "k--", linewidth=1, alpha=0.5, label="Invoice = Settled")
ax.set_xlim(0, max_val); ax.set_ylim(0, max_val)
ax.set_xlabel("Invoice Amount (KES)"); ax.set_ylabel("Settled Amount (KES)")
ax.set_title("Invoice vs Settled Amount", fontsize=14, fontweight="bold")
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/invoice_vs_settled.png", dpi=150)
plt.show()

In [ ]:
# Plot 4 — top 20 ICD codes (vague ones in red)
VAGUE_ICD = ["Z51.9", "Z00.0", "Z76.9", "Z71.9", "Z53.9"]
top_icd = df["ICD CODE"].value_counts().head(20)
colors = ["#D32F2F" if c in VAGUE_ICD else "#1976D2" for c in top_icd.index]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_icd.index[::-1], top_icd.values[::-1], color=colors[::-1])
ax.set_xlabel("Number of Claims")
ax.set_title("Top 20 ICD Codes (red = vague/catch-all)", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/icd_codes_top20.png", dpi=150)
plt.show()

## 7. Feature engineering

Build the 7 features the model will use. Each one is derived from a fraud-related hypothesis.

### Feature 1 — Invoice inflation ratio

How much the hospital billed versus what was actually paid. The strongest single signal in the dataset.

In [ ]:
df["SETTLED_SAFE"] = df["SETTLED AMOUNT"].replace(0, pd.NA)
df["invoice_inflation_ratio"] = df["INVOICE AMOUNT"] / df["SETTLED_SAFE"]

# Cap at 10x — anything higher is already obvious fraud and would distort the model
df["invoice_inflation_ratio"] = df["invoice_inflation_ratio"].clip(upper=10.0)

# Fill remaining nulls (where settled was 0) with the median
median_ratio = df["invoice_inflation_ratio"].median()
df["invoice_inflation_ratio"] = df["invoice_inflation_ratio"].fillna(median_ratio)

df["invoice_inflation_ratio"].describe()

### Feature 2 — Claim lag in days

Days between service date and claim submission. Long lags (months) suggest backdating.

In [ ]:
# Already computed earlier — just fill any remaining nulls with median
df["claim_lag_days"] = df["claim_lag_days"].fillna(df["claim_lag_days"].median())
df["claim_lag_days"].describe()

### Feature 3 — ICD code is vague

Binary flag: 1 if the diagnosis code is a catch-all (Z51.9, Z00.0, etc.). Vague codes on high-value claims are a known fraud signal.

In [ ]:
VAGUE_ICD_CODES = ["Z51.9", "Z00.0", "Z76.9", "Z71.9", "Z53.9"]
df["icd_is_vague"] = df["ICD CODE"].isin(VAGUE_ICD_CODES).astype(int)

print(f"Vague-ICD claims: {df['icd_is_vague'].sum():,} ({df['icd_is_vague'].mean():.1%})")
print(f"Codes checked   : {VAGUE_ICD_CODES}")

### Feature 4 — Provider average inflation

Per-provider mean of `invoice_inflation_ratio`. Catches providers who consistently overbill, even if individual claims look mild.

In [ ]:
provider_avg = (
    df.groupby("PROVIDER NAME")["invoice_inflation_ratio"]
    .mean()
    .rename("provider_avg_inflation")
)
df = df.merge(provider_avg, on="PROVIDER NAME", how="left")
df["provider_avg_inflation"] = df["provider_avg_inflation"].fillna(median_ratio)

# Top 10 worst providers
provider_avg.sort_values(ascending=False).head(10)

### Feature 5 — Provider claim count

Total claims submitted by this provider in the dataset. High volume isn't itself fraud, but combined with high inflation it is.

In [ ]:
provider_count = (
    df.groupby("PROVIDER NAME")["CLAIM NO"]
    .count()
    .rename("provider_claim_count")
)
df = df.merge(provider_count, on="PROVIDER NAME", how="left")
df["provider_claim_count"] = df["provider_claim_count"].fillna(1)
df["provider_claim_count"].describe()

### Feature 6 — Member claim frequency

Total claims per member. Unusually high frequency can indicate ghost beneficiaries or identity fraud.

In [ ]:
member_count = (
    df.groupby("MEMBER_HASH")["CLAIM NO"]
    .count()
    .rename("member_claim_count")
)
df = df.merge(member_count, on="MEMBER_HASH", how="left")
df["member_claim_count"] = df["member_claim_count"].fillna(1)

print(f"Median member claim count : {df['member_claim_count'].median():.0f}")
print(f"Max member claim count    : {df['member_claim_count'].max():.0f}")
print(f"Members with > 20 claims  : {(df['member_claim_count'] > 20).sum():,}")

### Feature 7 — Amount vs benefit benchmark

Ratio of the claim amount to the median for its benefit code. A 50K outpatient claim when the median is 4K is suspicious.

In [ ]:
benefit_median = (
    df.groupby("BENEFIT CODE")["INVOICE AMOUNT"]
    .median()
    .rename("benefit_median_amount")
)
df = df.merge(benefit_median, on="BENEFIT CODE", how="left")

df["amount_vs_benchmark"] = df["INVOICE AMOUNT"] / df["benefit_median_amount"]
df["amount_vs_benchmark"] = df["amount_vs_benchmark"].clip(upper=10.0).fillna(1.0)

print(f"Median ratio                  : {df['amount_vs_benchmark'].median():.3f}")
print(f"Claims > 3x benefit benchmark : {(df['amount_vs_benchmark'] > 3).sum():,}")

## 8. Proxy fraud label

Binary label for **evaluation only** — never fed to the model during training. A claim is suspicious if the insurer settled less than 80% of the invoice (meaning they already detected something off).

In [ ]:
df["proxy_fraud_label"] = (
    df["SETTLED AMOUNT"] < df["INVOICE AMOUNT"] * 0.80
).astype(int)

print(f"Suspicious (label=1): {df['proxy_fraud_label'].sum():,} ({df['proxy_fraud_label'].mean():.1%})")
print(f"Clean      (label=0): {(df['proxy_fraud_label']==0).sum():,}")
print(f"→ Use contamination={df['proxy_fraud_label'].mean():.2f} when training Isolation Forest")

## 9. Sanity check — does each feature actually relate to the proxy label?

If a feature shows the same fraud rate across all buckets, it carries no signal and can be dropped.

In [ ]:
# Fraud rate by claim lag bucket
lag_bins = pd.cut(df["claim_lag_days"],
                  bins=[0, 7, 30, 90, 180, 3300],
                  labels=["0-7d", "8-30d", "31-90d", "91-180d", ">180d"],
                  include_lowest=True)
print("Fraud rate by claim_lag_days bucket:")
print(df.groupby(lag_bins, observed=True)["proxy_fraud_label"].mean().apply(lambda x: f"{x:.1%}"))

In [ ]:
# Fraud rate by inflation ratio bucket — this is the strongest signal in the data
inf_bins = pd.cut(df["invoice_inflation_ratio"],
                  bins=[0, 1, 1.5, 3, 10],
                  labels=["=1", "1-1.5x", "1.5-3x", ">3x"],
                  include_lowest=True)
print("Fraud rate by invoice_inflation_ratio bucket:")
print(df.groupby(inf_bins, observed=True)["proxy_fraud_label"].mean().apply(lambda x: f"{x:.1%}"))

In [ ]:
# Fraud rate when ICD is vague vs specific
print("Fraud rate by icd_is_vague:")
print(df.groupby("icd_is_vague")["proxy_fraud_label"].mean().apply(lambda x: f"{x:.1%}"))

## 10. Save outputs for the model training phase

Two files leave this notebook:
- `claims_features.csv` — the 7 feature columns + the proxy label (fed to the model)
- `claims_full.csv` — the full anonymised dataset with features attached (used for joins later)

In [ ]:
FEATURE_COLUMNS = [
    "invoice_inflation_ratio",
    "claim_lag_days",
    "icd_is_vague",
    "provider_avg_inflation",
    "provider_claim_count",
    "member_claim_count",
    "amount_vs_benchmark",
]

# Confirm no nulls remain in the features
print("Nulls per feature:")
print(df[FEATURE_COLUMNS].isnull().sum())

In [ ]:
feature_out = FEATURE_COLUMNS + ["proxy_fraud_label"]
df[feature_out].to_csv(FEATURES_PATH, index=False)
print(f"Features saved → {FEATURES_PATH}")
print(f"Shape: {df[feature_out].shape}")

df.to_csv(FULL_OUTPUT_PATH, index=False)
print(f"Full dataset saved → {FULL_OUTPUT_PATH}")

## 11. Summary numbers to keep for the README

Copy these into the README's Problem Statement and Performance sections.

In [ ]:
print("=" * 60)
print("DATASET SUMMARY (for README)")
print("=" * 60)
print(f"Total claims          : {len(df):,}")
print(f"Date range            : {df['CLAIM DATE'].min().date()} → {df['CLAIM DATE'].max().date()}")
print(f"Providers             : {df['PROVIDER NAME'].nunique():,}")
print(f"Unique members        : {df['MEMBER_HASH'].nunique():,}")
print(f"Total invoiced (KES)  : {total_invoice:,.0f}")
print(f"Total settled  (KES)  : {total_settled:,.0f}")
print(f"Total variance (KES)  : {total_variance:,.0f} ({total_variance/total_invoice:.1%})")
print(f"Suspicious rate       : {df['proxy_fraud_label'].mean():.1%}")
print(f"Claims > 90 days lag  : {(df['claim_lag_days'] > 90).sum():,}")
print(f"Claims > 3x inflation : {(df['invoice_inflation_ratio'] > 3).sum():,}")
print(f"Vague-ICD claims      : {df['icd_is_vague'].sum():,}")
print("=" * 60)
print("Next: open the model training notebook (or run train.py)")